<a href="https://colab.research.google.com/github/yorgi544/data_acquisition/blob/main/akuisisi_openstreetmap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# LANGKAH 1: IMPORT LIBRARY

import requests
import pandas as pd
import json
import hashlib
import time
import os

from pathlib import Path
from datetime import datetime, timezone

print("Semua library berhasil dipanggil!")


Semua library berhasil dipanggil!


In [ ]:

# LANGKAH 2: IDENTITAS MAHASISWA

NAMA_PENGAMBIL = "Herkulano Yorgi Novriandri Simuk"
NIM_PENGAMBIL = "2501010203"
EMAIL_KAMPUS = "instiki@gamil.com"

print("Nama:", NAMA_PENGAMBIL)
print("NIM:", NIM_PENGAMBIL)
print("Email:", EMAIL_KAMPUS)


Nama: Herkulano Yorgi Novriandri Simuk
NIM: 2501010203
Email: instiki@gamil.com


In [ ]:

# LANGKAH 3: MEMBUAT FOLDER TUGAS

import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("docs", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)

print("Folder tugas berhasil dibuat!")
print("1. data/raw")
print("2. docs")
print("3. notebooks")


Folder tugas berhasil dibuat!
1. data/raw
2. docs
3. notebooks


In [ ]:

# LANGKAH 4: MENGHUBUNGKAN OPENSTREETMAP API

import requests
import json

ENDPOINT = "https://overpass-api.de/api/interpreter"

bbox = "-8.85,114.4,-8.0,115.75"

query = f"""
[out:json][timeout:25];
node["tourism"]({bbox});
out 5;
"""

headers = {
    "User-Agent": f"INSTIKI-AkuisisiData/1.0 ({NAMA_PENGAMBIL}; {NIM_PENGAMBIL}; {EMAIL_KAMPUS})"
}

response = requests.post(
    ENDPOINT,
    data={"data": query},
    headers=headers,
    timeout=60
)

print("Status HTTP:", response.status_code)

response.raise_for_status()

hasil = response.json()

print("Koneksi OpenStreetMap berhasil!")
print("Jumlah data percobaan:", len(hasil["elements"]))

print("\nContoh data:")
print(json.dumps(hasil["elements"][:2], indent=2))


Status HTTP: 200
Koneksi OpenStreetMap berhasil!
Jumlah data percobaan: 5

Contoh data:
[
  {
    "type": "node",
    "id": 253598001,
    "lat": -8.8098072,
    "lon": 115.1682183,
    "tags": {
      "artwork_type": "statue",
      "name": "Vishnu Statue",
      "tourism": "artwork"
    }
  },
  {
    "type": "node",
    "id": 260947786,
    "lat": -8.3385652,
    "lon": 115.6598117,
    "tags": {
      "name": "Coral View Cafe",
      "tourism": "chalet"
    }
  }
]


In [ ]:
# LANGKAH 5: MENGAMBIL DATA PARIWISATA BALI

import requests
import pandas as pd
import time
from datetime import datetime

# Wilayah Pulau Bali
bbox = "-8.85,114.4,-8.0,115.75"

# Query untuk mengambil objek pariwisata dan budaya
query = f"""
[out:json][timeout:90];
(
  node["tourism"]({bbox});
  way["tourism"]({bbox});
  relation["tourism"]({bbox});

  node["historic"]({bbox});
  way["historic"]({bbox});
  relation["historic"]({bbox});

  node["leisure"]({bbox});
  way["leisure"]({bbox});
  relation["leisure"]({bbox});

  node["amenity"~"^(museum|arts_centre|theatre)$"]({bbox});
  way["amenity"~"^(museum|arts_centre|theatre)$"]({bbox});
);
out center tags 500;
"""

# Daftar beberapa server Overpass API (sebagai cadangan)
ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter"
]

headers = {
    "User-Agent": f"INSTIKI-AkuisisiData/1.0 ({NAMA_PENGAMBIL}; {NIM_PENGAMBIL}; {EMAIL_KAMPUS})"
}

hasil = None

# Mencoba menghubungi endpoint hingga berhasil
for endpoint in ENDPOINTS:
    print(f"Mencoba mengambil data dari: {endpoint} ...")
    try:
        time.sleep(2) # Memberikan jeda sebelum request
        response = requests.post(
            endpoint,
            data={"data": query},
            headers=headers,
            timeout=120
        )

        if response.status_code == 200:
            hasil = response.json()
            print(f"Pengambilan data berhasil menggunakan {endpoint}!")
            print("Status HTTP:", response.status_code)
            print("Total objek dari API:", len(hasil["elements"]))
            break
        else:
            print(f"Gagal dengan status code: {response.status_code}. Mencoba endpoint berikutnya...")

    except Exception as e:
        print(f"Error saat menghubungi {endpoint}: {e}")
        print("Mencoba alternatif berikutnya...")

if hasil is None:
    raise RuntimeError("Semua server Overpass API gagal merespon. Silakan coba jalankan ulang sel ini beberapa saat lagi.")

Mencoba mengambil data dari: https://overpass-api.de/api/interpreter ...
Pengambilan data berhasil menggunakan https://overpass-api.de/api/interpreter!
Status HTTP: 200
Total objek dari API: 500


In [ ]:

# LANGKAH 6: MEMBUAT DATAFRAME

data = []

for item in hasil["elements"]:

    tags = item.get("tags", {})

    nama = tags.get("name")

    if not nama:
        continue

    latitude = item.get("lat")

    longitude = item.get("lon")

    if latitude is None:
        latitude = item.get("center", {}).get("lat")

    if longitude is None:
        longitude = item.get("center", {}).get("lon")

    data.append({
        "osm_type": item.get("type"),
        "osm_id": item.get("id"),
        "nama_tempat": nama,
        "tourism": tags.get("tourism"),
        "historic": tags.get("historic"),
        "leisure": tags.get("leisure"),
        "amenity": tags.get("amenity"),
        "latitude": latitude,
        "longitude": longitude
    })

df = pd.DataFrame(data)

df = df.drop_duplicates(
    subset=["osm_type", "osm_id"]
).reset_index(drop=True)

print("Total record:", len(df))
print("Total kolom:", len(df.columns))

display(df.head(10))


Total record: 463
Total kolom: 9


,osm_type,osm_id,nama_tempat,tourism,historic,leisure,amenity,latitude,longitude
0,node,253598001,Vishnu Statue,artwork,None,None,None,-8.809807,115.168218
1,node,260947786,Coral View Cafe,chalet,None,None,None,-8.338565,115.659812
2,node,275573942,Small Beautiful Waterfall,attraction,None,None,None,-8.175709,115.106360
3,node,316954328,Topi Inn,guest_house,None,None,None,-8.530209,115.511776
4,node,316955317,Puri Rai,hotel,None,None,None,-8.530337,115.509925
5,node,316956081,Pondok Wisata Serangan 2,guest_house,None,None,None,-8.529722,115.509134
6,node,316958748,Lombok Wisata,information,None,None,None,-8.531539,115.508546
7,node,316959068,Zen Inn,guest_house,None,None,None,-8.532257,115.507215
8,node,317207548,Artini 2,hotel,None,None,None,-8.514869,115.263688
9,node,317207553,Sania's House,guest_house,None,None,None,-8.508057,115.263001


In [ ]:

# LANGKAH 7: MENYIMPAN DATA KE CSV

from datetime import datetime
import os

tanggal = datetime.now().strftime("%Y%m%d")

nama_file = f"osm_pariwisata_bali_{tanggal}.csv"

lokasi_file = f"data/raw/{nama_file}"

df.to_csv(
    lokasi_file,
    index=False,
    encoding="utf-8"
)

print("DATA BERHASIL DISIMPAN!")
print("Nama file:", nama_file)
print("Lokasi:", lokasi_file)
print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))


DATA BERHASIL DISIMPAN!
Nama file: osm_pariwisata_bali_20261114.csv
Lokasi: data/raw/osm_pariwisata_bali_20261114.csv
Jumlah record: 463
Jumlah kolom: 9


In [ ]:

# LANGKAH 8: MEMERIKSA FILE CSV

import os

print("File tersedia:", os.path.exists(lokasi_file))

print("\nIsi folder data/raw:")
print(os.listdir("data/raw"))


File tersedia: True

Isi folder data/raw:
['osm_pariwisata_bali_20261114.csv']


In [ ]:

# LANGKAH 9: MEMBUAT PROVENANCE.JSON

import json
import hashlib
import os
from datetime import datetime, timezone

# Membaca file CSV
with open(lokasi_file, "rb") as file:
    isi_file = file.read()

# Menghitung ukuran file
ukuran_file = os.path.getsize(lokasi_file)

# Menghitung hash MD5
hash_md5 = hashlib.md5(isi_file).hexdigest()

# Membuat informasi provenance
provenance = {
    "nama_file": nama_file,
    "ukuran_file_bytes": ukuran_file,
    "hash_md5": hash_md5,
    "sumber_data": "OpenStreetMap melalui Overpass API",
    "penerbit": "OpenStreetMap contributors",
    "url_sumber": "https://overpass-api.de/api/interpreter",
    "lisensi": "Open Database License (ODbL)",
    "atribusi": "© OpenStreetMap contributors",
    "waktu_pembuatan_metadata_utc": datetime.now(timezone.utc).isoformat(),
    "jumlah_record": len(df),
    "jumlah_kolom": len(df.columns),
    "nama_pengambil": NAMA_PENGAMBIL,
    "nim_pengambil": NIM_PENGAMBIL
}

# Menyimpan file JSON
with open("docs/provenance.json", "w", encoding="utf-8") as file:
    json.dump(provenance, file, indent=4, ensure_ascii=False)

print("PROVENANCE BERHASIL DIBUAT!")
print(json.dumps(provenance, indent=4, ensure_ascii=False))


PROVENANCE BERHASIL DIBUAT!
{
    "nama_file": "osm_pariwisata_bali_20261114.csv",
    "ukuran_file_bytes": 31283,
    "hash_md5": "66aa6e10ad28788ab524c1234495f5a3",
    "sumber_data": "OpenStreetMap melalui Overpass API",
    "penerbit": "OpenStreetMap contributors",
    "url_sumber": "https://overpass-api.de/api/interpreter",
    "lisensi": "Open Database License (ODbL)",
    "atribusi": "© OpenStreetMap contributors",
    "waktu_pembuatan_metadata_utc": "2026-11-14T11:20:24.425289+00:00",
    "jumlah_record": 463,
    "jumlah_kolom": 9,
    "nama_pengambil": "Herkulano Yorgi Novriandri Simuk",
    "nim_pengambil": "2501010203"
}


In [ ]:

# LANGKAH 10: MEMERIKSA FILE PROVENANCE

import os
import json

with open("docs/provenance.json", "r", encoding="utf-8") as file:
    hasil_provenance = json.load(file)

print("FILE PROVENANCE BERHASIL DIBACA!")
print("Nama file:", hasil_provenance["nama_file"])
print("Jumlah record:", hasil_provenance["jumlah_record"])
print("Jumlah kolom:", hasil_provenance["jumlah_kolom"])
print("Hash MD5:", hasil_provenance["hash_md5"])

print("\nFile tersedia:",
      os.path.exists("docs/provenance.json"))


FILE PROVENANCE BERHASIL DIBACA!
Nama file: osm_pariwisata_bali_20261114.csv
Jumlah record: 463
Jumlah kolom: 9
Hash MD5: 66aa6e10ad28788ab524c1234495f5a3

File tersedia: True


In [ ]:

# LANGKAH 11: MEMBUAT CHECKLIST ETIKA DATA

import pandas as pd

ethics_data = [
    {
        "aspek": "Sumber Data",
        "pemeriksaan": "Menggunakan OpenStreetMap API",
        "status": "Sesuai"
    },
    {
        "aspek": "Lisensi",
        "pemeriksaan": "Memperhatikan lisensi Open Database License (ODbL)",
        "status": "Sesuai"
    },
    {
        "aspek": "Privasi",
        "pemeriksaan": "Tidak mengumpulkan data pribadi individu",
        "status": "Sesuai"
    },
    {
        "aspek": "Atribusi",
        "pemeriksaan": "Mencantumkan OpenStreetMap contributors",
        "status": "Sesuai"
    },
    {
        "aspek": "Keamanan",
        "pemeriksaan": "Tidak mengumpulkan informasi rahasia",
        "status": "Sesuai"
    },
    {
        "aspek": "Penggunaan Data",
        "pemeriksaan": "Data digunakan untuk keperluan akademik",
        "status": "Sesuai"
    }
]

df_ethics = pd.DataFrame(ethics_data)

df_ethics.to_csv(
    "docs/data_ethics_checklist.csv",
    index=False,
    encoding="utf-8"
)

print("CHECKLIST ETIKA BERHASIL DIBUAT!")

display(df_ethics)


CHECKLIST ETIKA BERHASIL DIBUAT!


,aspek,pemeriksaan,status
0,Sumber Data,Menggunakan OpenStreetMap API,Sesuai
1,Lisensi,Memperhatikan lisensi Open Database License (O...,Sesuai
2,Privasi,Tidak mengumpulkan data pribadi individu,Sesuai
3,Atribusi,Mencantumkan OpenStreetMap contributors,Sesuai
4,Keamanan,Tidak mengumpulkan informasi rahasia,Sesuai
5,Penggunaan Data,Data digunakan untuk keperluan akademik,Sesuai


In [ ]:

# LANGKAH 12: MEMERIKSA CHECKLIST ETIKA

df_ethics_check = pd.read_csv(
    "docs/data_ethics_checklist.csv"
)

print("File tersedia:",
      os.path.exists("docs/data_ethics_checklist.csv"))

print("Jumlah aspek:",
      len(df_ethics_check))

display(df_ethics_check)


File tersedia: True
Jumlah aspek: 6


,aspek,pemeriksaan,status
0,Sumber Data,Menggunakan OpenStreetMap API,Sesuai
1,Lisensi,Memperhatikan lisensi Open Database License (O...,Sesuai
2,Privasi,Tidak mengumpulkan data pribadi individu,Sesuai
3,Atribusi,Mencantumkan OpenStreetMap contributors,Sesuai
4,Keamanan,Tidak mengumpulkan informasi rahasia,Sesuai
5,Penggunaan Data,Data digunakan untuk keperluan akademik,Sesuai


In [ ]:


# LANGKAH 13: MEMBUAT README.MD

readme = """
# TUGAS 1
# Akuisisi Data Nyata melalui Web Scraping atau API

## Judul Proyek
Akuisisi Data Pariwisata Bali Menggunakan OpenStreetMap API

## Identitas Mahasiswa
- Nama: Herkulano Yorgi Novriandri Simuk
- NIM: 2501010203
- Program Studi: Informatika
- Institusi: INSTIKI

## 1. Deskripsi Proyek
Proyek ini bertujuan untuk mengumpulkan data
tempat pariwisata di wilayah Bali menggunakan
OpenStreetMap API melalui Overpass API.

## 2. Sumber Data
- Sumber: OpenStreetMap
- API: Overpass API
- URL: https://overpass-api.de/api/interpreter
- Lisensi: Open Database License (ODbL)
- Atribusi: © OpenStreetMap contributors

## 3. Metode Akuisisi
Data diperoleh menggunakan bahasa Python
di Google Colab dengan library Requests
dan Pandas.

Wilayah pengambilan data menggunakan
bounding box Pulau Bali.

Objek yang dikumpulkan meliputi:
- Tempat wisata
- Tempat bersejarah
- Area rekreasi
- Fasilitas budaya

## 4. Hasil Akuisisi
- Jumlah record: 463
- Jumlah kolom: 9
- Format data: CSV

## 5. Struktur Data
1. osm_type
2. osm_id
3. nama_tempat
4. tourism
5. historic
6. leisure
7. amenity
8. latitude
9. longitude

## 6. Struktur Folder
- data/raw/
  - osm_pariwisata_bali_20261114.csv
- docs/
  - provenance.json
  - data_ethics_checklist.csv
  - README.md

## 7. Dokumentasi
File provenance.json berisi informasi
sumber data, ukuran file, hash MD5,
lisensi, dan identitas pengambil data.

File data_ethics_checklist.csv berisi
pemeriksaan aspek etika penggunaan data.

## 8. Etika dan Lisensi
Penggunaan data memperhatikan lisensi
Open Database License (ODbL).
Atribusi OpenStreetMap contributors
dicantumkan sebagai sumber data.

Data digunakan untuk keperluan akademik.

## 9. Tools yang Digunakan
- Python
- Google Colab
- Requests
- Pandas
- OpenStreetMap API

## 10. Kesimpulan
Akuisisi data berhasil memperoleh 463 record
dengan 9 kolom dari OpenStreetMap API.
Data disimpan dalam format CSV dan dilengkapi
dengan dokumentasi provenance serta
checklist etika data.
"""

with open("docs/README.md", "w", encoding="utf-8") as file:
    file.write(readme)

print("README.md BERHASIL DIBUAT!")
print("Lokasi: docs/README.md")


README.md BERHASIL DIBUAT!
Lokasi: docs/README.md


In [ ]:

# LANGKAH 14: MEMERIKSA README

import os

print("File README tersedia:",
      os.path.exists("docs/README.md"))

with open("docs/README.md", "r", encoding="utf-8") as file:
    print(file.read())


File README tersedia: True

# TUGAS 1
# Akuisisi Data Nyata melalui Web Scraping atau API

## Judul Proyek
Akuisisi Data Pariwisata Bali Menggunakan OpenStreetMap API

## Identitas Mahasiswa
- Nama: Herkulano Yorgi Novriandri Simuk
- NIM: 2501010203
- Program Studi: Informatika
- Institusi: INSTIKI

## 1. Deskripsi Proyek
Proyek ini bertujuan untuk mengumpulkan data
tempat pariwisata di wilayah Bali menggunakan
OpenStreetMap API melalui Overpass API.

## 2. Sumber Data
- Sumber: OpenStreetMap
- API: Overpass API
- URL: https://overpass-api.de/api/interpreter
- Lisensi: Open Database License (ODbL)
- Atribusi: © OpenStreetMap contributors

## 3. Metode Akuisisi
Data diperoleh menggunakan bahasa Python
di Google Colab dengan library Requests
dan Pandas.

Wilayah pengambilan data menggunakan
bounding box Pulau Bali.

Objek yang dikumpulkan meliputi:
- Tempat wisata
- Tempat bersejarah
- Area rekreasi
- Fasilitas budaya

## 4. Hasil Akuisisi
- Jumlah record: 463
- Jumlah kolom: 9
- Format 

In [ ]:
import os

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [".config", "__pycache__"]]
    level = root.count(os.sep)
    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

./
    docs/
        provenance.json
        data_ethics_checklist.csv
        README.md
    notebooks/
    data/
        raw/
            osm_pariwisata_bali_20261114.csv
    sample_data/
        anscombe.json
        README.md
        california_housing_train.csv
        mnist_test.csv
        california_housing_test.csv
        mnist_train_small.csv
